# Gemini Manual Context-Retaining Batch Transcription Pipeline (Legacy)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/evaluate_gemini_manual_context.ipynb)

> [!IMPORTANT]
> **ARCHIVED / HISTORICAL BASELINE**
> This notebook represents the legacy, self-contained implementation for manual context-window ASR tracking. It does **not** use the shared `colabs.common` pipeline libraries. 
> 
> For active production runs, use the superior **Vertex AI Agent Sessions** context-aware pipeline in:
> `model/colabs/gemini_agent_session/`

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically checks for existing transcripts and skips already processed files to avoid duplicate work and optimize API costs (Delta processing).
2.  **Organization**: Results are written directly to GCS.


In [ ]:
%pip install -q google-genai loguru tqdm

In [ ]:
# @title Imports
import asyncio
from collections import defaultdict
import json
import os
from pathlib import Path
import re
import sys
import time
from urllib.parse import urlparse

from google import genai
from google.genai import types
from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")
PROJECT_NAME = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}

assert PROJECT_NAME, "PROJECT_NAME must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

GCS_INPUT_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
# @markdown Enable if modifications (ie, resampling/downmixing) were applied during segmentation:
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown Enable if masking was applied during segmentation:
AUDIO_MASKING = True  # @param {type:"boolean"}

assert not (AUDIO_PREPROCESSING and AUDIO_MASKING), (
    "Cannot enable both AUDIO_PREPROCESSING and AUDIO_MASKING simultaneously."
)

# Handle Preprocessing suffix
if not (AUDIO_PREPROCESSING or AUDIO_MASKING):
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"
if AUDIO_MASKING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_masked"


# Validation: Ensure required fields are filled to avoid IndexError in downstream GCS calls
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in the form above."
assert GCS_BUCKET, "GCS_BUCKET must be provided in the form above."


# Pipeline Control
OVERWRITE_EXISTING = True  # @param {type:"boolean"}

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
GCP_LOCATION = "global"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

SYSTEM_PROMPT = """
Evaluate all audio specifically as VHF/UHF fire-related dispatch radio traffic. The audio likely contains mic clicks, RF static, radio hum, and possibly some unintelligible speech. The speakers use heavy jargon.

EXPECTED TERMINOLOGY:
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript exactly as said, with no newlines.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. Format all unit identifiers as the unit type followed by digits (e.g., Engine 41, Battalion 2).
4. Do not continue the speech segment beyond what is spoken.
5. QUALITY GATE: Transcribe only what you hear with high acoustic certainty. If a portion of audio is obscured, noisy, or ambiguous, you MUST replace that specific portion with [UNINTELLIGIBLE]. Do not attempt to phonetically guess ambiguous noise.

TASK:
Transcribe the attached audio. Output strictly the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
}

TURN_PROMPT = """COMMAND: The preceding audio turns are for situational awareness ONLY. DO NOT re-transcribe them.
Execute strict verbatim transcription EXCLUSIVELY on the single audio clip attached to this specific message.
Apply all CRITICAL RULES."""

# Size of context history:
# MAX_HISTORY_ITEMS = N full turns (N/2 user audio turns + N/2 model text turns)
MAX_HISTORY_ITEMS = 10  # @param {type:"integer"}

CONCURRENCY_LIMIT = 20
MAX_RETRIES = 3

# fmt: off
LOG_LEVEL = "WARNING"  # @param ["DEBUG", "INFO", "WARNING", "ERROR"]
# fmt: on

logger.remove()
logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level=LOG_LEVEL
)

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent (Run Once)
# This is a one-time environment setup step. If you have already configured
# permissions for this bucket and project, you can skip this cell.
#
# To run, uncomment the lines below:
#
# !gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
#     --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
#     --role="roles/storage.objectViewer"


In [ ]:
# @title Execute Rolling-Window Transcription Pipeline


# Initialize standard clients
client = genai.Client(
    vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION
)
storage_client = storage.Client(project=GCP_PROJECT_ID)

CHECKPOINT_FILE = "interim_backup_predictions.jsonl"
checkpoint_write_lock = asyncio.Lock()


def get_gcs_checkpoint_blob():
    """Derives GCS checkpoint path consistently for both load and upload."""
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    checkpoint_blob_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE
    return storage_client.bucket(out_bucket).blob(checkpoint_blob_path)


def load_gcs_checkpoint() -> dict[str, str]:
    blob = get_gcs_checkpoint_blob()
    records = {}

    if blob.exists():
        logger.info(
            f"Found existing checkpoint on GCS: {blob.name}. Loading..."
        )
        blob.download_to_filename(CHECKPOINT_FILE)

        # Parse the file immediately and return the dictionary
        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record["transcript"]
        logger.info(f"Loaded {len(records)} completed records from GCS.")
    else:
        logger.info("No remote checkpoint found. Starting fresh.")

    return records


async def upload_checkpoint_to_gcs() -> None:
    """Synchronizes the local checkpoint file to GCS to protect against runtime disconnects."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            blob = get_gcs_checkpoint_blob()
            blob.upload_from_filename(CHECKPOINT_FILE)
        except Exception as sync_err:
            logger.warning(f"Failed to backup checkpoint to GCS: {sync_err}")


async def process_single_channel(
    channel_id: str,
    segment_entries: list[dict],
    completed_records: dict[str, str],
    semaphore: asyncio.Semaphore,
    pbar: tqdm,
) -> list[dict]:
    """Processes a channel using true multi-turn Audio->Text pairs for context."""
    results = []

    async with semaphore:
        logger.info(
            f"Starting rolling window for {channel_id} ({len(segment_entries)} segments)"
        )
        transcript_history = []  # Will store dicts of {"uri": ..., "text": ...}

        for entry in segment_entries:
            uri = entry["audio_filepath"]

            if uri in completed_records:
                cached_transcript = completed_records[uri]
                results.append(
                    {
                        "example_id": channel_id,
                        "audio_filepath": uri,
                        "transcript": cached_transcript,
                        "error": None,
                    }
                )

                if cached_transcript.strip() != "[UNINTELLIGIBLE]":
                    transcript_history.append(
                        {"uri": uri, "text": cached_transcript}
                    )

                if len(transcript_history) > MAX_HISTORY_ITEMS:
                    transcript_history = transcript_history[-MAX_HISTORY_ITEMS:]
                pbar.update(1)
                continue

            # Build the payload
            payload_contents = []

            # 1. Inject rolling history as explicit User (Audio) -> Model (Text) pairs
            for past_turn in transcript_history:
                payload_contents.append(
                    {
                        "role": "user",
                        "parts": [
                            {
                                "file_data": {
                                    "file_uri": past_turn["uri"],
                                    "mime_type": "audio/flac",
                                }
                            }
                        ],
                    }
                )
                payload_contents.append(
                    {"role": "model", "parts": [{"text": past_turn["text"]}]}
                )

            # 2. Append the current request.
            # Put the text prompt BEFORE the audio so the model reads the constraint first.
            current_user_turn = {
                "role": "user",
                "parts": [
                    {"text": TURN_PROMPT},
                    {"file_data": {"file_uri": uri, "mime_type": "audio/flac"}},
                ],
            }
            payload_contents.append(current_user_turn)

            transcript = None
            error_msg = "Unknown error"

            try:
                logger.debug(f"[API CALL] Sending request for {uri}...")
                response = await client.aio.models.generate_content(
                    model=MODEL_ID,
                    contents=payload_contents,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT.strip(),
                        safety_settings=SAFETY_SETTINGS,
                        temperature=GENERATION_CONFIG["temperature"],
                        max_output_tokens=GENERATION_CONFIG[
                            "max_output_tokens"
                        ],
                    ),
                )

                if response.candidates and response.candidates[0].content.parts:
                    transcript = response.text.strip()
                    logger.success(
                        f"[API SUCCESS] Received transcript for {uri}"
                    )
                else:
                    finish_reason = getattr(
                        response.candidates[0], "finish_reason", "UNKNOWN"
                    )
                    error_msg = (
                        f"Response blocked/empty. Reason: {finish_reason}"
                    )
                    logger.warning(
                        f"[API BLOCKED] {uri} - Reason: {finish_reason}"
                    )

            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)}"
                logger.error(f"[API ERROR] {uri} - {error_msg}")

            if transcript:
                result_dict = {
                    "example_id": channel_id,
                    "audio_filepath": uri,
                    "transcript": transcript,
                    "error": None,
                }
                results.append(result_dict)

                async with checkpoint_write_lock:
                    with open(CHECKPOINT_FILE, "a") as f:
                        f.write(json.dumps(result_dict) + "\n")
                    # Incremental GCS Backup to prevent data loss on runtime disconnects
                    try:
                        await upload_checkpoint_to_gcs()
                    except Exception as sync_err:
                        logger.warning(
                            f"Failed to backup checkpoint to GCS: {sync_err}"
                        )

                if transcript.strip() != "[UNINTELLIGIBLE]":
                    transcript_history.append({"uri": uri, "text": transcript})

                if len(transcript_history) > MAX_HISTORY_ITEMS:
                    transcript_history = transcript_history[-MAX_HISTORY_ITEMS:]
            else:
                results.append(
                    {
                        "example_id": channel_id,
                        "audio_filepath": uri,
                        "transcript": None,
                        "error": error_msg,
                    }
                )

            pbar.update(1)

    return results


async def main() -> None:
    # 1. Initialize empty records
    completed_records = {}

    # 2. Logic Fork: Wipe vs Resume
    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs.")
        # Wipe GCS Output
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        # Clean local
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
    else:
        # Load from GCS so we know what's already done
        logger.info(
            "OVERWRITE_EXISTING is False. Resuming from GCS checkpoint..."
        )
        completed_records = load_gcs_checkpoint()

    # 3. Load manifest
    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest not found at {MANIFEST_URI}")

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0
    for line in content:
        if line.strip():
            entry = json.loads(line)
            channels[entry["example_id"]].append(entry)
            total_segments += 1

    for ch in channels:
        channels[ch].sort(key=lambda x: x.get("offset", 0))

    # 4. Filter only for what is actually missing
    active_channels = {}
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries

    if not active_channels:
        logger.info("Everything complete.")
    else:
        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        with tqdm(
            total=total_segments, desc="Processing Transcriptions"
        ) as pbar:
            tasks = [
                process_single_channel(
                    cid, entries, completed_records, semaphore, pbar
                )
                for cid, entries in active_channels.items()
            ]
            await asyncio.gather(*tasks)

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            final_ndjson = f.read()
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            final_ndjson
        )
        logger.info(f"Success. Results saved to {CONSISTENT_OUTPUT_URI}")

    # Premium Visual summary receipt table
    if completed_records:
        logger.info(
            f"Pipeline Complete. Recorded {len(completed_records)} transcripts."
        )
        df = pd.DataFrame(
            list(completed_records.items()),
            columns=["audio_filepath", "transcript"],
        )
        df["example_id"] = df["audio_filepath"].apply(
            lambda x: Path(x).parent.name
        )
        display(df[["example_id", "audio_filepath", "transcript"]].head(10))


await main()

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")

if expected_count == actual_count:
    print("\n✅ SUCCESS: All segments were transcribed and recorded.")
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )